<center><h1>Li_Wenxin_HW3</h1></center>
<br>
<br>

Name: Wenxin Li
<br>
Github Username: Wendyllli
<br>
USC ID: 1269534541

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
from pathlib import Path
import hashlib
import re
import sys
import numpy as np
import pandas as pd

SEED = 552
N_BOOTSTRAP = 20_000
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.precision", 6)
print(f"Python {sys.version.split()[0]}, NumPy {np.__version__}, pandas {pd.__version__}")

Python 3.14.3, NumPy 2.5.4, pandas 3.0.6


Get the AReM Data Set

In [2]:
ACTIVITIES = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
CHANNELS = ["avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
EXPECTED_COUNTS = {"bending1": 7, "bending2": 6, **{a: 15 for a in ACTIVITIES[2:]}}

DATA_DIR = Path("data/AReM")
if not DATA_DIR.is_dir():
    DATA_DIR = Path("C:/Users/Wendy/Desktop/DSCI552HW/Homework 3, 4 Data/AReM")
if not DATA_DIR.is_dir():
    raise FileNotFoundError("Set DATA_DIR to the AReM folder containing the seven activity folders.")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

def dataset_number(path):
    match = re.fullmatch(r"dataset(\d+)\.csv", path.name)
    if match is None:
        raise ValueError(f"Unexpected filename: {path.name}")
    return int(match.group(1))

def read_recording(path):
    rows = []
    whitespace_rows = 0
    trailing_comma_rows = 0
    for line_number, raw_line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), 1):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if "," in line:
            fields = [value.strip() for value in line.split(",")]
            if fields[-1] == "":
                trailing_comma_rows += 1
                while fields and fields[-1] == "":
                    fields.pop()
        else:
            fields = line.split()
            whitespace_rows += 1
        if len(fields) != 7 or any(value == "" for value in fields):
            raise ValueError(f"{path}, line {line_number}: expected time and six valid measurements")
        rows.append([float(value) for value in fields])
    values = np.asarray(rows, dtype=float)
    if values.ndim != 2 or values.shape[1] != 7 or not np.isfinite(values).all():
        raise ValueError(f"Invalid data in {path}")
    timestamps = values[:, 0]
    if len(np.unique(timestamps)) != len(timestamps) or np.any(np.diff(timestamps) <= 0):
        raise ValueError(f"Duplicated or unordered timestamps in {path}")
    expected_times = np.arange(0, 120_000, 250)
    if not np.isin(timestamps, expected_times).all():
        raise ValueError(f"Unexpected timestamp in {path}")
    missing_times = sorted(set(expected_times) - set(timestamps))
    audit = {
        "n_observations": len(values),
        "whitespace_rows": whitespace_rows,
        "trailing_comma_rows": trailing_comma_rows,
        "missing_timestamps_ms": ";".join(str(int(t)) for t in missing_times),
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    }
    return pd.DataFrame(values[:, 1:], columns=CHANNELS, index=timestamps), audit

recordings = {}
metadata_rows = []
for activity in ACTIVITIES:
    files = sorted((DATA_DIR / activity).glob("dataset*.csv"), key=dataset_number)
    assert len(files) == EXPECTED_COUNTS[activity]
    assert [dataset_number(p) for p in files] == list(range(1, EXPECTED_COUNTS[activity] + 1))
    for path in files:
        instance_id = f"{activity}/{path.name}"
        series, audit = read_recording(path)
        recordings[instance_id] = series
        metadata_rows.append({"instance_id": instance_id, "activity": activity,
                              "dataset_number": dataset_number(path), **audit})
metadata = pd.DataFrame(metadata_rows).set_index("instance_id")
metadata.insert(0, "instance", np.arange(1, len(metadata) + 1))
assert len(recordings) == 88
assert metadata["n_observations"].value_counts().to_dict() == {480: 87, 479: 1}
assert metadata["n_observations"].sum() == 42_239
print(f"Loaded {len(recordings)} recordings; {metadata['n_observations'].sum():,} observed time rows.")
print("Observed row counts:", metadata["n_observations"].value_counts().to_dict())
metadata.loc[(metadata["n_observations"] != 480) | (metadata["whitespace_rows"] > 0) |
             (metadata["trailing_comma_rows"] > 0),
             ["activity", "n_observations", "whitespace_rows", "trailing_comma_rows", "missing_timestamps_ms"]]


Loaded 88 recordings; 42,239 observed time rows.
Observed row counts: {480: 87, 479: 1}


,activity,n_observations,whitespace_rows,trailing_comma_rows,missing_timestamps_ms
instance_id,,,,,
bending2/dataset4.csv,bending2,480,480,0,
cycling/dataset9.csv,cycling,480,0,1,
cycling/dataset14.csv,cycling,480,0,1,
sitting/dataset8.csv,sitting,479,0,0,13500


### (b) Test and Train Data

In [3]:
is_bending = metadata["activity"].isin(["bending1", "bending2"])
test_mask = (is_bending & (metadata["dataset_number"] <= 2)) | (~is_bending & (metadata["dataset_number"] <= 3))
metadata["split"] = np.where(test_mask, "test", "train")
train_ids = metadata.index[metadata["split"] == "train"]
test_ids = metadata.index[metadata["split"] == "test"]
assert len(train_ids) == 69 and len(test_ids) == 19
assert set(train_ids).isdisjoint(test_ids)
assert set(train_ids) | set(test_ids) == set(metadata.index)
split_summary = pd.crosstab(metadata["activity"], metadata["split"]).reindex(ACTIVITIES)[["train", "test"]]
split_summary["total"] = split_summary.sum(axis=1)
split_summary.loc["TOTAL"] = split_summary.sum(axis=0)
metadata.to_csv(RESULTS_DIR / "recording_manifest.csv")
split_summary.to_csv(RESULTS_DIR / "split_summary.csv")
split_summary


split,train,test,total
activity,,,
bending1,5,2,7
bending2,4,2,6
cycling,12,3,15
lying,12,3,15
sitting,12,3,15
standing,12,3,15
walking,12,3,15
TOTAL,69,19,88


### (c) Feature Extraction

#### i. Research


| Category | Examples | Information captured |
|---|---|---|
| Level and position | Mean, median, minimum, maximum, quartiles | Typical value and extremes |
| Spread | Standard deviation, variance, range, interquartile range | Amplitude variability |
| Distribution shape | Skewness, kurtosis | Asymmetry and tail behavior |
| Magnitude | Root mean square, sum of squared values | Signal magnitude and energy |
| Temporal behavior | Lagged autocorrelation, linear trend, mean absolute successive change, peak counts, threshold crossings | Persistence, drift, local changes, and events |

#### ii. Extraction

For each recording, I calculated the minimum, maximum, mean, median, sample standard deviation, first quartile, and third quartile for each of the six time series. This gives 42 features per recording and an 88 × 42 feature matrix. I used `ddof=1` for the sample standard deviation.

In [4]:
STATISTICS = ["min", "max", "mean", "median", "std", "q1", "q3"]
FEATURE_COLUMNS = [f"{stat}_{j}" for j in range(1, 7) for stat in STATISTICS]

def extract_features(series):
    result = {}
    for j, channel in enumerate(CHANNELS, 1):
        x = series[channel].to_numpy(dtype=float)
        q1, q3 = np.quantile(x, [0.25, 0.75], method="linear")
        values = [x.min(), x.max(), x.mean(), np.median(x), x.std(ddof=1), q1, q3]
        result.update({f"{stat}_{j}": float(value) for stat, value in zip(STATISTICS, values)})
    return result

features = pd.DataFrame.from_dict({key: extract_features(value) for key, value in recordings.items()}, orient="index")
features.index.name = "instance_id"
features = features.loc[metadata.index, FEATURE_COLUMNS]
X_train, X_test = features.loc[train_ids].copy(), features.loc[test_ids].copy()
y_train, y_test = metadata.loc[train_ids, "activity"].copy(), metadata.loc[test_ids, "activity"].copy()
assert features.shape == (88, 42)
assert X_train.shape == (69, 42) and X_test.shape == (19, 42)
assert np.isfinite(features.to_numpy()).all()
for j in range(1, 7):
    ordered = features[[f"{name}_{j}" for name in ["min", "q1", "median", "q3", "max"]]].to_numpy()
    assert (np.diff(ordered, axis=1) >= 0).all()
    assert (features[f"std_{j}"] >= 0).all()
features.to_csv(RESULTS_DIR / "features_88x42.csv")
metadata[["instance", "activity", "split", "n_observations"]].join(features).to_csv(RESULTS_DIR / "features_labeled.csv")
X_train.to_csv(RESULTS_DIR / "X_train.csv")
X_test.to_csv(RESULTS_DIR / "X_test.csv")
y_train.to_csv(RESULTS_DIR / "y_train.csv")
y_test.to_csv(RESULTS_DIR / "y_test.csv")
print("Feature matrix:", features.shape)
print("Train:", X_train.shape, "Test:", X_test.shape)
features.head()


Feature matrix: (88, 42)
Train: (69, 42) Test: (19, 42)


,min_1,max_1,mean_1,median_1,std_1,q1_1,q3_1,min_2,max_2,mean_2,median_2,std_2,q1_2,q3_2,min_3,max_3,mean_3,median_3,std_3,q1_3,q3_3,min_4,max_4,mean_4,median_4,std_4,q1_4,q3_4,min_5,max_5,mean_5,median_5,std_5,q1_5,q3_5,min_6,max_6,mean_6,median_6,std_6,q1_6,q3_6
instance_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
bending1/dataset1.csv,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,1.30,0.358604,0.43,0.322605,0.00,0.50,4.0,29.50,19.040937,19.250,4.462952,16.00,23.2500,0.0,7.23,0.832542,0.50,0.965659,0.00,1.120,27.25,38.25,34.311292,35.0,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
bending1/dataset2.csv,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,0.0,1.22,0.372437,0.47,0.289158,0.00,0.50,2.0,29.50,20.096979,21.000,3.893737,19.00,22.2500,0.0,5.76,0.876438,0.50,1.047259,0.00,1.145,27.67,38.50,33.024583,33.0,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
bending1/dataset3.csv,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,0.47,0.338690,0.00,0.50,6.5,29.75,22.122354,23.000,3.030943,19.75,24.0000,0.0,4.44,0.497312,0.43,0.550657,0.00,0.830,29.00,38.50,35.588458,36.0,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
bending1/dataset4.csv,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,0.50,0.630860,0.00,1.12,8.5,30.00,22.183625,23.000,3.810469,20.50,24.3725,0.0,5.15,0.989917,0.83,0.953730,0.43,1.300,20.00,38.67,33.493917,35.0,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
bending1/dataset5.csv,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,0.50,0.405469,0.43,0.71,3.0,28.25,19.006563,19.125,4.087107,16.50,22.0625,0.0,6.42,0.841875,0.50,0.928801,0.43,1.120,23.67,37.50,29.857083,30.0,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


In [5]:
first_recording = recordings["bending1/dataset1.csv"]
example_table = pd.DataFrame({channel: [features.loc["bending1/dataset1.csv", f"{stat}_{j}"]
                                      for stat in STATISTICS]
                              for j, channel in enumerate(CHANNELS, 1)}, index=STATISTICS)
example_table.index.name = "statistic"
example_table


,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23
statistic,,,,,,
min,37.250000,0.000000,4.000000,0.000000,27.250000,0.000000
max,45.000000,1.300000,29.500000,7.230000,38.250000,1.920000
mean,40.624792,0.358604,19.040937,0.832542,34.311292,0.570583
median,40.500000,0.430000,19.250000,0.500000,35.000000,0.430000
std,1.476967,0.322605,4.462952,0.965659,2.188449,0.582915
q1,39.250000,0.000000,16.000000,0.000000,33.000000,0.000000
q3,42.000000,0.500000,23.250000,1.120000,36.000000,1.300000


#### iii. Standard Deviation

I calculated the sample standard deviation of each feature across the 88 recordings. To find its 90% confidence interval, I randomly sampled 88 rows with replacement and recalculated the standard deviations. I repeated this 20,000 times, using a random seed of 552. For each feature, I used the 5th and 95th percentiles of the bootstrap results as the interval endpoints.

In [6]:
def bootstrap_feature_std(frame, n_bootstrap=20_000, seed=552, batch_size=512):
    values = frame.to_numpy(dtype=float)
    n, p = values.shape
    rng = np.random.default_rng(seed)
    bootstrap_std = np.empty((n_bootstrap, p), dtype=float)
    for start in range(0, n_bootstrap, batch_size):
        stop = min(start + batch_size, n_bootstrap)
        indices = rng.integers(0, n, size=(stop - start, n))
        bootstrap_std[start:stop] = values[indices].std(axis=1, ddof=1)
    lower, upper = np.quantile(bootstrap_std, [0.05, 0.95], axis=0, method="linear")
    table = pd.DataFrame({
        "feature_std": values.std(axis=0, ddof=1),
        "ci_90_lower": lower,
        "ci_90_upper": upper,
    }, index=frame.columns)
    table.index.name = "feature"
    return table, bootstrap_std

std_ci, bootstrap_replicates = bootstrap_feature_std(features, N_BOOTSTRAP, SEED)
assert std_ci.shape == (42, 3)
assert np.isfinite(std_ci.to_numpy()).all()
assert (std_ci["ci_90_lower"] >= 0).all()
assert (std_ci["ci_90_upper"] >= std_ci["ci_90_lower"]).all()
std_ci.to_csv(RESULTS_DIR / "feature_std_90ci.csv")
print(f"Percentile bootstrap: N={len(features)}, B={N_BOOTSTRAP:,}, seed={SEED}, sample SD (ddof=1)")
std_ci


Percentile bootstrap: N=88, B=20,000, seed=552, sample SD (ddof=1)


,feature_std,ci_90_lower,ci_90_upper
feature,,,
min_1,9.569975,8.270275,10.758378
max_1,4.394362,3.321867,5.294858
mean_1,5.335718,4.707926,5.881131
median_1,5.440054,4.793776,5.996089
std_1,1.772153,1.571332,1.951606
q1_1,6.153590,5.582096,6.645435
q3_1,5.138925,4.331936,5.840724
min_2,0.000000,0.000000,0.000000
max_2,5.062729,4.625041,5.407043


These intervals show uncertainty in each feature’s population standard deviation, not its mean or individual values. The standard deviation of the bootstrap estimates gives the standard error.

A constant feature gives a bootstrap interval of [0, 0], but this does not prove it is constant in the population. Results may vary with the random seed, calculation method, and handling of missing data.

#### iv. Select Features

I chose mean, standard deviation, and maximum. The mean shows the average signal level, the standard deviation measures variation, and the maximum captures peak values. Together, they may help distinguish different activities.

Using these three statistics for all six time series gives 18 features. This choice is based on my judgment; model testing would be needed to check how well they work.

In [7]:
SELECTED_STATISTICS = ["mean", "std", "max"]
selected_columns = [f"{stat}_{j}" for j in range(1, 7) for stat in SELECTED_STATISTICS]
assert len(selected_columns) == 18
X_train[selected_columns].to_csv(RESULTS_DIR / "X_train_selected18.csv")
X_test[selected_columns].to_csv(RESULTS_DIR / "X_test_selected18.csv")
training_summary = X_train[["mean_1", "std_1", "max_1"]].join(y_train).groupby("activity").median().reindex(ACTIVITIES)
training_summary.columns = ["median of mean_1", "median of std_1", "median of max_1"]
training_summary.to_csv(RESULTS_DIR / "training_selected_feature_summary.csv")
training_summary

,median of mean_1,median of std_1,median of max_1
activity,,,
bending1,43.454958,1.618364,47.750
bending2,30.998177,5.533969,46.500
cycling,36.753406,3.734644,44.330
lying,39.587010,0.861206,40.835
sitting,42.321427,1.701255,45.875
standing,43.453771,2.289928,47.415
walking,34.509708,4.747038,46.000


## 2. ISLR 3.7.4

### (a) Linear Train

The cubic model will have a training RSS less than or equal to that of the linear model. It includes the linear model as a special case, so the extra terms cannot increase the minimum training RSS.

### (b) Linear Test

I would generally expect the linear model to have a lower test RSS because the true relationship is linear. The extra terms in the cubic model may fit noise and increase variance without reducing bias.

### (c) Not Linear Train

The cubic model will still have a training RSS less than or equal to that of the linear model. This holds regardless of the true relationship because the cubic model includes all the terms in the linear model.

### (d) Not Linear Testing

There is not enough information to tell which model will have a lower test RSS. The cubic model may capture the nonlinear relationship better, but its extra flexibility also increases variance. The result depends on the true relationship and the noise level.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 